# Naive RAG Walkthrough
투자 트랜스크립트 -> pgvector -> gpt-4o-mini.

In [ ]:
from naive_rag.loader import load_transcripts
docs = load_transcripts(limit=3)
print(f"Loaded {len(docs)} documents")
docs[0].metadata, docs[0].page_content[:300]

## Split

In [ ]:
from naive_rag.splitter import split_documents
chunks = split_documents(docs)
print(f"{len(docs)} docs -> {len(chunks)} chunks")
chunks[0].page_content[:200], chunks[0].metadata

## Embed (single query for inspection)

In [ ]:
from naive_rag.embeddings import build_embeddings
emb = build_embeddings()
vec = emb.embed_query("매크로 지표")
print(f"dim={len(vec)}, first 5={vec[:5]}")

## Retrieve

In [ ]:
from naive_rag.retriever import build_retriever
retriever = build_retriever()
hits = retriever.invoke("최근 자주 인용되는 매크로 지표는?")
for i, h in enumerate(hits):
    print(f"--- hit {i} (channel={h.metadata.get('channel')}) ---")
    print(h.page_content[:200])

## Generate

In [ ]:
from naive_rag.chain import build_chain
chain = build_chain()
answer = chain.invoke("최근 트랜스크립트에서 자주 인용되는 매크로 지표는?")
print(answer)

## Run all 10 eval questions

In [ ]:
questions = [
    "최근 트랜스크립트에서 언급된 주요 매수 근거는 무엇인가?",
    "자주 인용되는 매크로 지표는 무엇인가?",
    "반증 조건(thesis가 깨지는 조건)으로 제시된 내용은?",
    "특정 종목에 대한 방향성(상승/하락)과 시간 지평은?",
    "최근 콘텐츠에서 위험 요소로 언급된 것은?",
    "특정 섹터(반도체/2차전지/AI 등)에 대한 시각은?",
    "두 인플루언서의 의견이 갈리는 지점은?",
    "동일 종목에 대해 서로 다른 근거를 제시한 사례는?",
    "미국 금리/연준 정책에 대한 견해는?",
    "환율(원/달러)에 대한 언급과 그 영향은?",
]
for i, q in enumerate(questions, 1):
    print(f"\n=== Q{i}. {q}")
    print(chain.invoke(q))